In [0]:
# ORDERS INTEGRATED
# 1️⃣ Canonical Status Definitions
# Load silver-layer orders, payments, and order items tables

df_orders = spark.table("retail_catalog.silver.orders")
df_payments = spark.table("retail_catalog.silver.payments")
df_order_items = spark.table("retail_catalog.silver.orderitems")

In [0]:
# Aggregate payments at the order level
# For each order, calculate number of payments, total payment value, and unique payment types
# Add a boolean 'has_payment' for downstream business logic

from pyspark.sql.functions import count, sum, col, when, collect_set, collect_list, size

payments_agg = (
    df_payments
    .groupBy("order_id")
    .agg(
        count("*").alias("number_of_payments"), # count payment records for each order
        sum("payment_value").alias("total_payment_value"),
        collect_set("payment_type").alias("payment_types_used")
    )
    .withColumn("has_payment", col("number_of_payments") > 0)
)

# display(payments_agg)

In [0]:
# Aggregate order items for each order
# Compute item count, sum of item prices, sum of shipping charges, and list of product IDs
# Add a boolean 'has_items' for downstream business logic

order_items_agg = (
    df_order_items
    .groupBy("order_id")
    .agg(
        count("*").alias("item_count"),
        sum("price").alias("items_price_total"),
        sum("shipping_charges").alias("shipping_total"),
        collect_list("product_id").alias("products")
    )
    .withColumn("has_items", col("item_count") > 0)
)

# display(order_items_agg)

In [0]:
# Combine orders with their aggregated payment and item information
# Use left joins to retain all orders even if payments or items are missing

orders_financial_base = (
    df_orders
    .join(payments_agg, "order_id", "left")
    .join(order_items_agg, "order_id", "left")
)

# display(orders_financial_base)

In [0]:
# Define canonical status columns for business logic
# - Fill null 'has_payment'/'has_items' with False
# - Mark paid/delivered/completed with business rules

orders_canonical = (
    orders_financial_base
    .withColumn("has_payment", when(col("has_payment").isNull(), False).otherwise(col("has_payment")))
    .withColumn("has_items", when(col("has_items").isNull(), False).otherwise(col("has_items")))
    .withColumn("is_paid", col("total_payment_value") > 0)
    .withColumn("is_delivered", when((col("order_status") == "delivered") & col("order_delivered_customer_date").isNotNull(), True).otherwise(False))
    .withColumn("is_completed", col("is_paid") & col("is_delivered"))
)

# display(orders_canonical)

In [0]:
# Construct unified business status for each order using canonical flags
# Business status is categorized as 'completed', 'paid_not_delivered', 'created_not_paid', or 'invalid'

orders_canonical = orders_canonical.withColumn(
    "order_business_status",
    when(col("is_completed"), "completed")
    .when(col("is_paid") & ~col("is_delivered"), "paid_not_delivered")
    .when(col("has_items") & ~col("has_payment"), "created_not_paid")
    .otherwise("invalid")
)

# display(orders_canonical)

In [0]:
# 2️⃣ Revenue & Financial Normalization
# Compute expected order value and payment gap
# 'payment_matches_items' True if gap < 1.0 (tolerance for minor rounding errors)

from pyspark.sql.functions import col, abs, when, round

orders_financial = orders_canonical.withColumn("expected_order_value", round(col("items_price_total") + col("shipping_total"), 2)) \
                                        .withColumn("order_value_gap", round(col("total_payment_value") - col("expected_order_value"), 2)) \
                                        .withColumn("payment_matches_items", when(abs(col("order_value_gap")) < 1.0, True).otherwise(False)) 


# orders_financial.limit(5).display()

In [0]:
# Select and organize all relevant columns for gold/orders_integrated
# Includes identifiers, status flags, item and payment details, normalized financials, timestamps, and delivery/completion info

orders_integrated = orders_financial.select(
    # Identifiers
    "order_id", "customer_id",
    # Status
    "order_status", "order_business_status",
    # Item details
    "item_count", "items_price_total", "shipping_total", "has_items",
    # Payment details
    "number_of_payments", "total_payment_value", "payment_types_used", "has_payment", "is_paid", "payment_matches_items",
    # Financial calculations
    "expected_order_value", "order_value_gap",
    # Timestamps (process order)
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
    # Delivery and completion flags
    "is_delivered", "is_completed"
)

# display(orders_integrated.limit(10))

In [0]:
# Produce final integrated orders DataFrame for persistence or downstream analytics
orders_integrated

DataFrame[order_id: string, customer_id: string, order_status: string, order_business_status: string, item_count: bigint, items_price_total: double, shipping_total: double, has_items: boolean, number_of_payments: bigint, total_payment_value: double, payment_types_used: array<string>, has_payment: boolean, is_paid: boolean, payment_matches_items: boolean, expected_order_value: double, order_value_gap: double, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, is_delivered: boolean, is_completed: boolean]